# Run all: execute the complete experiment suite

Executes the final experimental notebooks and `load_results.ipynb` in dependency order, each in a fresh kernel. Executed copies with outputs are written under `runs/<run-version>/`; source notebooks are not modified.

Set `QUICK = True` for the reduced execution check. QUICK outputs use a separate run version from the production results.

In [ ]:
from pathlib import Path
import os
import time
import datetime
import threading
import sys
import nbformat
from nbclient import NotebookClient
from nbclient.exceptions import CellExecutionError

HERE = Path.cwd().resolve()
if not (HERE / "shared_utils.py").exists():
    raise RuntimeError("Run run_all.ipynb from the experiments directory containing shared_utils.py.")

ORDER = [
    "0_preliminaries.ipynb",
    "1_core_model.ipynb",
    "2_benchmark_decomposition.ipynb",
    "4_calibration.ipynb",
    "5_supplementary_nulls.ipynb",
    "6_common_factor_diagnostic.ipynb",
    "7_sparsity_sweep.ipynb",
    "8_ragnar_search.ipynb",
    "9_common_component.ipynb",
    "10_simulation_study.ipynb",
    "11_rolling_origins.ipynb",
    # "load_results.ipynb",
]
QUICK = False
STOP_ON_FAIL = True
KERNEL = "python3"
TIMEOUT = 4 * 3600

stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
run_version = f"{stamp}_quick" if QUICK else stamp
os.environ["GNAR_RUN_VERSION"] = run_version
os.environ["GNAR_QUICK"] = "1" if QUICK else "0"

results_dir = HERE / "results" / run_version
outputs_dir = HERE / "outputs" / run_version
run_dir = HERE / "runs" / run_version
if results_dir.exists() or outputs_dir.exists() or run_dir.exists():
    raise FileExistsError(f"Run version {run_version!r} already exists; restart to create a fresh version.")
run_dir.mkdir(parents=True, exist_ok=False)

# Compile every code cell before starting any expensive fit.
for name in ORDER:
    path = HERE / name
    if not path.exists():
        raise FileNotFoundError(path)
    notebook = nbformat.read(path, as_version=4)
    for index, cell in enumerate(notebook.cells):
        if cell.cell_type != "code":
            continue
        try:
            compile(cell.source, f"{name}:cell{index}", "exec")
        except SyntaxError as exc:
            raise SyntaxError(f"Syntax error in {name}, cell {index}: {exc}") from exc

print(f"run version: {run_version}")
print(f"QUICK={QUICK}")
print(f"executed copies -> {run_dir}\n")
print("all notebook code cells compile")

run version: 20260825_034651_quick
QUICK=True
executed copies -> /Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/experiments/runs/20260825_034651_quick

all notebook code cells compile


In [3]:
def _fmt_secs(elapsed):
    """Format seconds as h/m/s, dropping empty leading units."""
    elapsed = int(elapsed)
    hours, rem = divmod(elapsed, 3600)
    minutes, seconds = divmod(rem, 60)
    return (f"{hours}h{minutes:02d}m{seconds:02d}s" if hours
            else f"{minutes}m{seconds:02d}s" if minutes else f"{seconds}s")


def _heading_before(notebook, index):
    """Return the nearest markdown heading at or above a cell index."""
    for j in range(index, -1, -1):
        cell = notebook.cells[j]
        if cell.cell_type == "markdown":
            for line in cell.source.splitlines():
                text = line.strip().lstrip("#").strip()
                if text:
                    return text[:60]
    return "(no heading)"


class _Ticker:
    """Background thread that overwrites one line with a live elapsed timer."""
    def __init__(self, label):
        self.label = label
        self.t0 = time.time()
        self.on = True
        self.thread = threading.Thread(target=self._loop, daemon=True)
        self.thread.start()

    def _loop(self):
        while self.on:
            elapsed = time.time() - self.t0
            sys.stdout.write(f"\r    {self.label}  [{_fmt_secs(elapsed)}]     ")
            sys.stdout.flush()
            time.sleep(1)

    def stop(self, mark):
        self.on = False
        self.thread.join(timeout=2)
        elapsed = time.time() - self.t0
        sys.stdout.write(f"\r    {mark} {self.label}  ({_fmt_secs(elapsed)})            \n")
        sys.stdout.flush()

In [4]:
def run_one(name):
    """Execute one notebook in a fresh kernel and save its executed copy."""
    source = HERE / name
    if not source.exists():
        return {"nb": name, "status": "MISSING", "secs": 0.0,
                "cell": None, "err": "file not found"}

    notebook = nbformat.read(source, as_version=4)
    client = NotebookClient(
        notebook,
        timeout=TIMEOUT,
        kernel_name=KERNEL,
        resources={"metadata": {"path": str(HERE)}},
    )
    t0 = time.time()
    status, failed_cell, error = "OK", None, None
    code_cells = [i for i, cell in enumerate(notebook.cells) if cell.cell_type == "code"]

    try:
        with client.setup_kernel():
            for number, index in enumerate(code_cells, 1):
                label = f"cell {number}/{len(code_cells)} - {_heading_before(notebook, index)}"
                ticker = _Ticker(label)
                try:
                    client.execute_cell(notebook.cells[index], index)
                    ticker.stop("[ok]")
                except CellExecutionError as exc:
                    ticker.stop("[x]")
                    status, failed_cell = "FAIL", index
                    error = str(exc).strip().splitlines()[-1][:300]
                    break
    except Exception as exc:
        status, failed_cell = "ERROR", None
        error = f"{type(exc).__name__}: {exc}"[:300]
    finally:
        nbformat.write(notebook, run_dir / name)

    return {
        "nb": name, "status": status, "secs": time.time() - t0,
        "cell": failed_cell, "err": error,
    }

In [5]:
summary = []
for name in ORDER:
    print(f"> {name}")
    result = run_one(name)
    summary.append(result)

    if result["status"] == "OK":
        print(f"  [ok] {name} complete ({_fmt_secs(result['secs'])})\n")
    elif result["status"] == "FAIL":
        print(f"  [x] {name} FAILED at code cell #{result['cell']}: {result['err']}\n")
    elif result["status"] == "ERROR":
        print(f"  [x] {name} ERROR: {result['err']}\n")
    else:
        print(f"  [x] {name} MISSING: {result['err']}\n")

    if STOP_ON_FAIL and result["status"] != "OK":
        print("STOP_ON_FAIL=True, halting at the first unsuccessful notebook.")
        break

> 0_preliminaries.ipynb
    [ok] cell 1/7 - Notebook 0: pre-evaluation specification selection  (1s)            
    [ok] cell 2/7 - Complete-network lag screen  (3s)            
    [ok] cell 3/7 - Network, density and stage search  (5s)            
    [ok] cell 4/7 - Geographic k=2 lag refinement  (3s)            
    [ok] cell 5/7 - Stage-depth complexity diagnostic  (1s)            
    [ok] cell 6/7 - Selection confirmation  (6s)            
    [ok] cell 7/7 - Save results  (1s)            
  [ok] 0_preliminaries.ipynb complete (21s)

> 1_core_model.ipynb
    [ok] cell 1/14 - Notebook 1: primary high-order models  (4s)            
    [ok] cell 2/14 - Model fits  (1s)            
    [ok] cell 3/14 - Model fits  (6s)            
    [ok] cell 4/14 - Model fits  (2s)            
    [ok] cell 5/14 - Model fits  (3s)            
    [ok] cell 6/14 - Model fits  (3s)            
    [ok] cell 7/14 - Model fits  (3s)            
    [ok] cell 8/14 - Model fits  (4s)            
    

In [6]:
import pandas as pd

summary_df = pd.DataFrame(summary)[["nb", "status", "secs", "cell", "err"]]
summary_df["time"] = summary_df["secs"].map(_fmt_secs)
print(summary_df[["nb", "status", "time", "cell", "err"]].to_string(index=False))

n_ok = int((summary_df["status"] == "OK").sum())
all_passed = n_ok == len(ORDER)
print(f"\n{'ALL PASSED' if all_passed else 'RUN INCOMPLETE'}  |  "
      f"{n_ok}/{len(ORDER)} OK  |  total {_fmt_secs(summary_df['secs'].sum())}")
print(f"run version: {run_version}")
print(f"executed notebooks saved under: {run_dir}")

if not all_passed:
    raise RuntimeError("The run did not complete successfully; do not treat this version as results of record.")

                              nb status  time cell  err
           0_preliminaries.ipynb     OK   21s None None
              1_core_model.ipynb     OK   36s None None
 2_benchmark_decomposition.ipynb     OK 2m34s None None
             4_calibration.ipynb     OK    3s None None
6_common_factor_diagnostic.ipynb     OK    4s None None
     5_supplementary_nulls.ipynb     OK   17s None None
          7_sparsity_sweep.ipynb     OK   15s None None
           8_ragnar_search.ipynb     OK   10s None None
       10_simulation_study.ipynb     OK 3m26s None None
        11_rolling_origins.ipynb     OK   20s None None
    9_factor_adjusted_gnar.ipynb     OK 1m34s None None
              load_results.ipynb     OK   16s None None

ALL PASSED  |  12/12 OK  |  total 10m01s
run version: 20260825_034651_quick
executed notebooks saved under: /Users/patrickgunn/Documents/Unis/Imperial College/Research Proj/experiments/runs/20260825_034651_quick
